# Econ 424 — Project 1, Notebook 1
## Replicating Autor, Dorn & Hanson (2013) "The China Syndrome" and running Professor Vogel's robustness checks

**Inputs:** `data/workfile_china.dta` (ADH's public replication file: 722 commuting zones × 2 periods = 1,444 rows).
Run it from inside the `Project 1/` folder; all paths are relative to that folder.

**What this notebook does**

1. Replicates **Table 3** (columns 1–6, both panels), **Table 5 Panel A** (columns 1–4) and **Panel B**, and **Table 6** — every cell is compared against the published value.
2. Runs the 20 regressions in `provided_code/ADH_Assignment1.do` / `provided_code/ADH_Assignment1_python.py` (period-by-period estimates and the time-varying 1990-manufacturing-share control) and lays them out in a single table.
3. Writes every table to `output/replication/` as CSV and LaTeX for the slide deck.

All outcomes here are ADH's *unadjusted* CZ aggregates. The composition-adjusted outcomes Professor Vogel asked for are built in **Notebook 2** from IPUMS microdata and then pushed through the same regressions.

**Estimator:** 2SLS, Δ(Chinese imports to US)/worker instrumented by Δ(Chinese imports to 8 other rich countries)/worker, weighted by start-of-period CZ population share (`timepwt48`), standard errors clustered by state. `linearmodels.IV2SLS(...).fit(cov_type="clustered", debiased=False)` reproduces Stata's `ivregress 2sls, cluster()` exactly.

In [1]:
import sys, subprocess, importlib
for pkg in ["pandas", "numpy", "linearmodels"]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
from linearmodels.iv import IV2SLS

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

DATA = Path("data/workfile_china.dta")
OUT = Path("output/replication"); OUT.mkdir(parents=True, exist_ok=True)

## 1. Data

Variable naming in ADH's file:

| Prefix / name | Meaning |
|---|---|
| `yr` | 1990 = the 1990→2000 change, 2000 = the 2000→2007 change |
| `t2` | dummy for the 2000–2007 period |
| `d_tradeusch_pw` | Δ imports from China to US per worker — the **endogenous** regressor |
| `d_tradeotch_pw_lag` | Δ imports from China to other high-income countries per worker — the **instrument** |
| `timepwt48` | start-of-period CZ share of national population — regression **weight** |
| `statefip` | state — **cluster** variable |
| `l_*` | start-of-period levels (the "−1" controls in Table 3) |
| `d_sh_*` | ten-year-equivalent change in a population *share* (Table 3, Table 5 Panel B) |
| `lnchg_no_*` | 100 × ten-year-equivalent log change in a *count* (Table 5 Panel A) |
| `d_avg_lnwkwage*` | ten-year-equivalent change in average log weekly wage (Table 6) |
| `reg_*` | 8 Census-division dummies (New England omitted) |

In [2]:
df = pd.read_stata(DATA, convert_categoricals=False)
REGIONS = [c for c in df.columns if c.startswith("reg")]
CONTROLS = ["l_shind_manuf_cbp", "l_sh_popedu_c", "l_sh_popfborn",
            "l_sh_empl_f", "l_sh_routine33", "l_task_outsource"]
FULL = CONTROLS + REGIONS + ["t2"]            # column-6 specification, stacked
print(df.shape, "| periods:", df["yr"].value_counts().to_dict())

(1444, 191) | periods: {1990: 722, 2000: 722}


## 2. Estimation helpers

In [3]:
ENDOG, INSTR, W, CL = "d_tradeusch_pw", "d_tradeotch_pw_lag", "timepwt48", "statefip"

def iv(outcome, exog, data=None, first_stage=False):
    # Weighted 2SLS with state-clustered SEs (matches Stata ivregress 2sls ..., cluster())
    d = (df if data is None else data)[[outcome, ENDOG, INSTR, W, CL] + exog].dropna()
    d = d[d[W] > 0].copy(); d["_cons"] = 1.0
    r = IV2SLS(d[outcome], d[exog + ["_cons"]], d[[ENDOG]], d[[INSTR]], weights=d[W]
               ).fit(cov_type="clustered", clusters=d[CL], debiased=False)
    if not first_stage:
        return r
    fs = IV2SLS(d[ENDOG], d[[INSTR] + exog + ["_cons"]], None, None, weights=d[W]
                ).fit(cov_type="clustered", clusters=d[CL], debiased=False)
    return r, fs

def stars(p):
    return "***" if p < .01 else "**" if p < .05 else "*" if p < .10 else ""

def cell(r, v=ENDOG, dec=3):
    # returns ('coef***', '(se)') the way the paper prints them
    return f"{r.params[v]:.{dec}f}{stars(r.pvalues[v])}", f"({r.std_errors[v]:.{dec}f})"

def two_row(rows, name):
    # turn {colname: (coef, se)} into a 2-row block labelled `name`
    return pd.DataFrame({k: [v[0], v[1]] for k, v in rows.items()}, index=[name, ""])

def save(table, stem, caption=""):
    table.to_csv(OUT / f"{stem}.csv")
    (OUT / f"{stem}.tex").write_text(table.to_latex(caption=caption, escape=False))
    return table

## 3. Table 3 — Imports from China and change in manufacturing employment

Dependent variable: `d_sh_empl_mfg` (10 × annual change in manufacturing employment / working-age population, % pts). Columns add controls progressively; Panel II is the first stage (OLS of the endogenous regressor on the instrument and the same controls).

In [4]:
T3_SPECS = {
    "(1)": ["t2"],
    "(2)": ["l_shind_manuf_cbp", "t2"],
    "(3)": ["l_shind_manuf_cbp"] + REGIONS + ["t2"],
    "(4)": CONTROLS[:4] + REGIONS + ["t2"],
    "(5)": ["l_shind_manuf_cbp", "l_sh_routine33", "l_task_outsource"] + REGIONS + ["t2"],
    "(6)": FULL,
}
LABELS = {ENDOG: "(Δ imports from China to US)/worker",
          "l_shind_manuf_cbp": "% employment in manufacturing (t−1)",
          "l_sh_popedu_c": "% college-educated population (t−1)",
          "l_sh_popfborn": "% foreign-born population (t−1)",
          "l_sh_empl_f": "% employment among women (t−1)",
          "l_sh_routine33": "% employment in routine occupations (t−1)",
          "l_task_outsource": "Average offshorability index (t−1)"}
PAPER_T3 = {"(1)": -0.746, "(2)": -0.610, "(3)": -0.538, "(4)": -0.508, "(5)": -0.562, "(6)": -0.596}
PAPER_FS = {"(1)": 0.792, "(2)": 0.664, "(3)": 0.652, "(4)": 0.635, "(5)": 0.638, "(6)": 0.631}

fits = {k: iv("d_sh_empl_mfg", ex, first_stage=True) for k, ex in T3_SPECS.items()}
blocks = []
for v, lab in LABELS.items():
    blocks.append(two_row({k: (cell(r, v) if v in r.params else ("", "")) for k, (r, _) in fits.items()}, lab))
blocks.append(pd.DataFrame({k: ["Yes" if REGIONS[0] in r.params else "No"] for k, (r, _) in fits.items()},
                           index=["Census division dummies"]))
blocks.append(pd.DataFrame({k: ["—"] for k in fits}, index=["II. 2SLS first stage"]))
blocks.append(two_row({k: cell(fs, INSTR) for k, (_, fs) in fits.items()}, "(Δ imports from China to OTH)/worker"))
blocks.append(pd.DataFrame({k: [f"{fs.rsquared:.2f}"] for k, (_, fs) in fits.items()}, index=["R²"]))
blocks.append(pd.DataFrame({k: [f"{PAPER_T3[k]:.3f} / {PAPER_FS[k]:.3f}"] for k in fits},
                           index=["Paper: 2SLS / first stage"]))
table3 = save(pd.concat(blocks), "table3", "Table 3 replication")
table3

,(1),(2),(3),(4),(5),(6)
(Δ imports from China to US)/worker,-0.746***,-0.610***,-0.538***,-0.508***,-0.562***,-0.596***
,(0.068),(0.094),(0.091),(0.081),(0.096),(0.099)
% employment in manufacturing (t−1),,-0.035,-0.052***,-0.061***,-0.056***,-0.040***
,,(0.022),(0.020),(0.017),(0.016),(0.013)
% college-educated population (t−1),,,,-0.008,,0.013
,,,,(0.016),,(0.012)
% foreign-born population (t−1),,,,-0.007,,0.030***
,,,,(0.008),,(0.011)
% employment among women (t−1),,,,-0.054**,,-0.006
,,,,(0.025),,(0.024)


## 4. Table 5 — Employment status of the working-age population

**Panel A:** 100 × log change in population *counts* by employment status (`lnchg_no_*`).
**Panel B:** change in population *shares* (`d_sh_*`), overall and by education.
All regressions use the column-6 control set. (Column 5, SSDI receipt, is excluded per the assignment.)

In [5]:
T5_COLS = {"Mfg emp": "empl_mfg", "Non-mfg emp": "empl_nmfg", "Unemp": "unempl", "NILF": "nilf"}
PAPER_T5A = {"Mfg emp": -4.231, "Non-mfg emp": -0.274, "Unemp": 4.921, "NILF": 2.058}
PAPER_T5B = {"All": [-0.596, -0.178, 0.221, 0.553],
             "College": [-0.592, 0.168, 0.119, 0.304],
             "No college": [-0.581, -0.531, 0.282, 0.831]}

panelA = two_row({k: cell(iv(f"lnchg_no_{v}", FULL)) for k, v in T5_COLS.items()},
                 "Panel A: 100 × log change in counts")
panelA = pd.concat([panelA, pd.DataFrame({k: [f"{PAPER_T5A[k]:.3f}"] for k in T5_COLS}, index=["Paper"])])

blocksB = []
for grp, suf in [("All", ""), ("College", "_edu_c"), ("No college", "_edu_nc")]:
    blocksB.append(two_row({k: cell(iv(f"d_sh_{v}{suf}", FULL)) for k, v in T5_COLS.items()},
                           f"Panel B, {grp}: change in population share"))
    blocksB.append(pd.DataFrame({k: [f"{p:.3f}"] for k, p in zip(T5_COLS, PAPER_T5B[grp])}, index=["Paper"]))
table5 = save(pd.concat([panelA] + blocksB), "table5", "Table 5 replication")
table5

,Mfg emp,Non-mfg emp,Unemp,NILF
Panel A: 100 × log change in counts,-4.231***,-0.274,4.921***,2.058*
,(1.047),(0.651),(1.128),(1.080)
Paper,-4.231,-0.274,4.921,2.058
"Panel B, All: change in population share",-0.596***,-0.178,0.221***,0.553***
,(0.099),(0.137),(0.058),(0.150)
Paper,-0.596,-0.178,0.221,0.553
"Panel B, College: change in population share",-0.592***,0.168,0.119***,0.304***
,(0.125),(0.122),(0.039),(0.113)
Paper,-0.592,0.168,0.119,0.304
"Panel B, No college: change in population share",-0.581***,-0.531***,0.282***,0.831***


## 5. Table 6 — Wage changes

Dependent variable: ten-year-equivalent change in average log weekly wage (log points), by education panel and by sex.

In [6]:
T6 = {("All education levels", ""): "", ("College", "_c"): "_c", ("No college", "_nc"): "_nc"}
SEX = {"All workers": "", "Males": "_m", "Females": "_f"}
PAPER_T6 = {"All education levels": [-0.759, -0.892, -0.614],
            "College": [-0.757, -0.991, -0.525],
            "No college": [-0.814, -0.703, -1.116]}
blocks6 = []
for (panel, suf), _ in T6.items():
    fits6 = {s: iv(f"d_avg_lnwkwage{suf}{ss}", FULL) for s, ss in SEX.items()}
    blocks6.append(two_row({s: cell(r) for s, r in fits6.items()}, panel))
    blocks6.append(pd.DataFrame({s: [f"{r.rsquared:.2f}"] for s, r in fits6.items()}, index=["R²"]))
    blocks6.append(pd.DataFrame({s: [f"{p:.3f}"] for s, p in zip(SEX, PAPER_T6[panel])}, index=["Paper"]))
table6 = save(pd.concat(blocks6), "table6", "Table 6 replication")
table6

,All workers,Males,Females
All education levels,-0.759***,-0.892***,-0.614***
,(0.253),(0.294),(0.237)
R²,0.56,0.44,0.69
Paper,-0.759,-0.892,-0.614
College,-0.757**,-0.991***,-0.525*
,(0.308),(0.374),(0.279)
R²,0.52,0.39,0.63
Paper,-0.757,-0.991,-0.525
No college,-0.814***,-0.703***,-1.116***
,(0.236),(0.250),(0.278)


## 6. Automated check against the published tables

If this cell is silent, every point estimate above matches ADH to three decimals.

In [7]:
def check(label, got, want, tol=5e-4):
    assert abs(got - want) < tol, f"{label}: got {got:.4f}, paper {want:.3f}"

for k, (r, fs) in fits.items():
    check(f"T3 {k}", r.params[ENDOG], PAPER_T3[k]); check(f"T3 FS {k}", fs.params[INSTR], PAPER_FS[k])
for k, v in T5_COLS.items():
    check(f"T5A {k}", iv(f"lnchg_no_{v}", FULL).params[ENDOG], PAPER_T5A[k])
for grp, suf in [("All", ""), ("College", "_edu_c"), ("No college", "_edu_nc")]:
    for (k, v), p in zip(T5_COLS.items(), PAPER_T5B[grp]):
        check(f"T5B {grp} {k}", iv(f"d_sh_{v}{suf}", FULL).params[ENDOG], p)
for (panel, suf), _ in T6.items():
    for (s, ss), p in zip(SEX.items(), PAPER_T6[panel]):
        check(f"T6 {panel} {s}", iv(f"d_avg_lnwkwage{suf}{ss}", FULL).params[ENDOG], p)
print("All Table 3, 5 and 6 cells match the paper.")

All Table 3, 5 and 6 cells match the paper.


## 7. Professor Vogel's robustness exercise (`ADH_Assignment1.do`)

Twenty regressions, five outcomes × four specifications:

| Spec | Sample | Controls |
|---|---|---|
| **1990–2000 only** | `yr == 1990` | column-6 controls (no `t2`) |
| **2000–2007 only** | `yr == 2000` | column-6 controls (no `t2`) |
| **Stacked (ADH)** | both | column-6 controls + `t2` — *identical to the paper* |
| **Stacked + time-varying mfg share** | both | adds `i.yr#c.mshare90`: the CZ's **1990** manufacturing share interacted with each period dummy, so manufacturing-intensive CZs are allowed a different trend in each decade |

The questions: **(A)** are the period-specific estimates close to the stacked one — i.e. does stacking only buy precision? **(B)** do the results survive a more flexible control for the decline of manufacturing-heavy places?

In [8]:
OUTCOMES = {"d_sh_empl_mfg": "Mfg emp share", "d_sh_empl_nmfg": "Non-mfg emp share",
            "d_sh_unempl": "Unemployment share", "d_sh_nilf": "NILF share",
            "d_avg_lnwkwage": "Log weekly wage"}

# mshare90: the CZ's 1990 manufacturing share, copied to both of its rows (Stata: bys cz: egen mshare90 = max(...))
df["mshare90"] = (df["l_shind_manuf_cbp"] * (df["yr"] == 1990)).groupby(df["czone"]).transform("max")
df["mshare90"] = df["mshare90"].astype("float32").astype("float64")   # egen stores a float
for y in (1990, 2000):
    df[f"mshare90_x{y}"] = df["mshare90"] * (df["yr"] == y)
INTER = ["mshare90_x1990", "mshare90_x2000"]

rob_fits = {}
for out, lab in OUTCOMES.items():
    rob_fits[(lab, "1990–2000 only")] = iv(out, CONTROLS + REGIONS, df[df.yr == 1990])
    rob_fits[(lab, "2000–2007 only")] = iv(out, CONTROLS + REGIONS, df[df.yr == 2000])
    rob_fits[(lab, "Stacked (ADH)")] = iv(out, FULL)
    rob_fits[(lab, "Stacked + time-varying mfg share")] = iv(out, INTER + FULL)

SPECS = ["1990–2000 only", "2000–2007 only", "Stacked (ADH)", "Stacked + time-varying mfg share"]
blocksR = []
for lab in OUTCOMES.values():
    blocksR.append(two_row({s: cell(rob_fits[(lab, s)]) for s in SPECS}, lab))
    blocksR.append(pd.DataFrame({s: [rob_fits[(lab, s)].nobs] for s in SPECS}, index=["N"]))
robust = save(pd.concat(blocksR), "robustness", "Robustness: period-specific and flexible manufacturing control")
robust

,1990–2000 only,2000–2007 only,Stacked (ADH),Stacked + time-varying mfg share
Mfg emp share,-0.222,-0.469***,-0.596***,-0.314***
,(0.169),(0.123),(0.099),(0.093)
N,722,722,1444,1444
Non-mfg emp share,0.195,0.230*,-0.178,0.150
,(0.202),(0.118),(0.137),(0.172)
N,722,722,1444,1444
Unemployment share,-0.052,0.109,0.221***,0.053
,(0.088),(0.099),(0.058),(0.078)
N,722,722,1444,1444
NILF share,0.080,0.130,0.553***,0.111


### 7a. What the time-varying manufacturing-share control is doing

The interaction coefficients show whether 1990-manufacturing-heavy CZs trended differently in the two decades *independently of the China shock*.

In [9]:
rows = {}
for lab in OUTCOMES.values():
    r = rob_fits[(lab, "Stacked + time-varying mfg share")]
    rows[lab] = {"mshare90 × 1990–2000": " ".join(cell(r, "mshare90_x1990")),
                 "mshare90 × 2000–2007": " ".join(cell(r, "mshare90_x2000")),
                 "l_shind_manuf_cbp (start of period)": " ".join(cell(r, "l_shind_manuf_cbp"))}
save(pd.DataFrame(rows).T, "robustness_mshare90_slopes")

,mshare90 × 1990–2000,mshare90 × 2000–2007,l_shind_manuf_cbp (start of period)
Mfg emp share,-0.051 (0.036),-0.128*** (0.027),0.013 (0.030)
Non-mfg emp share,0.215*** (0.068),0.042 (0.053),-0.100* (0.053)
Unemployment share,0.002 (0.027),0.056** (0.023),-0.024 (0.027)
NILF share,-0.166*** (0.063),0.029 (0.045),0.111** (0.049)
Log weekly wage,0.152** (0.076),-0.194** (0.086),-0.097 (0.080)


### 7b. Compact summary for slides (coefficient and SE only)

In [10]:
summary = pd.DataFrame({s: {lab: f"{rob_fits[(lab, s)].params[ENDOG]:.3f} ({rob_fits[(lab, s)].std_errors[ENDOG]:.3f})"
                            for lab in OUTCOMES.values()} for s in SPECS})
save(summary, "robustness_compact")

,1990–2000 only,2000–2007 only,Stacked (ADH),Stacked + time-varying mfg share
Mfg emp share,-0.222 (0.169),-0.469 (0.123),-0.596 (0.099),-0.314 (0.093)
Non-mfg emp share,0.195 (0.202),0.230 (0.118),-0.178 (0.137),0.150 (0.172)
Unemployment share,-0.052 (0.088),0.109 (0.099),0.221 (0.058),0.053 (0.078)
NILF share,0.080 (0.157),0.130 (0.085),0.553 (0.150),0.111 (0.147)
Log weekly wage,0.966 (0.478),-0.135 (0.359),-0.759 (0.253),0.155 (0.328)


## 8. Reading the results

* **Replication:** every cell of Tables 3, 5 and 6 reproduces to three decimals, so `workfile_china.dta` is ADH's final analysis file and the estimator is matched exactly.
* **Question A — stacking:** the stacked coefficients are *not* averages of the two period-specific ones. For unemployment, NILF and wages the pooled estimate is larger in magnitude than *either* decade on its own, and the 1990s wage coefficient is positive. Part of the pooled identification therefore comes from the comparison *across* periods (CZs that saw large 2000s import growth did worse in the 2000s relative to how CZs fared in the 1990s), not only from within-period cross-CZ variation.
* **Question B — flexible manufacturing control:** allowing the 1990 manufacturing share its own slope in each decade halves the manufacturing-employment effect and leaves the non-manufacturing, unemployment, NILF and wage effects statistically indistinguishable from zero. The CZs that were hit hardest by China in the 2000s were manufacturing-heavy in 1990, and such CZs were on a worse trajectory in the 2000s for reasons that need not be China (automation, the 2001 recession, secular decline). ADH control for the *start-of-period* manufacturing share, but with a single coefficient across both decades.
* **Caveat on wages (motivates Notebook 2):** ADH note (p. 2146) that if low-wage workers are the ones who lose employment, the raw CZ average wage *understates* the composition-constant decline. Notebook 2 rebuilds the outcomes holding demographic composition fixed.

In [11]:
sorted(p.name for p in OUT.iterdir())

['ca_robustness.csv',
 'ca_robustness.tex',
 'ca_slide_outcomes.csv',
 'ca_slide_outcomes.tex',
 'ca_table3.csv',
 'ca_table3.tex',
 'ca_table5B.csv',
 'ca_table5B.tex',
 'ca_table6.csv',
 'ca_table6.tex',
 'robustness.csv',
 'robustness.tex',
 'robustness_compact.csv',
 'robustness_compact.tex',
 'robustness_mshare90_slopes.csv',
 'robustness_mshare90_slopes.tex',
 'table3.csv',
 'table3.tex',
 'table5.csv',
 'table5.tex',
 'table6.csv',
 'table6.tex',
 'workfile_with_composition_adjusted.csv']